In [18]:
import pandas as pd
import os
from ortools.linear_solver import pywraplp
from datetime import time as dt_time


In [19]:
DATA_PATH = r"../data/Raw Data/Data.xlsx"

Rooms = pd.read_excel(DATA_PATH, sheet_name="Rooms")
Courses = pd.read_excel(DATA_PATH, sheet_name="Courses")
Doctors = pd.read_excel(DATA_PATH, sheet_name="Doctors")
Divisions = pd.read_excel(DATA_PATH, sheet_name="Division")

print("Data Loaded Successfully")
print("Courses:", len(Courses))
print("Rooms:", len(Rooms))
print("Doctors:", len(Doctors))
print("Divisions:", len(Divisions))


Data Loaded Successfully
Courses: 53
Rooms: 8
Doctors: 81
Divisions: 10


In [20]:
def time_to_minutes(t):

    if pd.isna(t):
        return 0

    if isinstance(t, dt_time):
        return t.hour * 60 + t.minute

    if hasattr(t, "hour") and hasattr(t, "minute"):
        return t.hour * 60 + t.minute

    if isinstance(t, (int, float)):
        return int(t)

    if isinstance(t, str):
        parts = t.strip().split(":")
        return int(parts[0]) * 60 + int(parts[1])

    return 0


def overlap(s1, e1, s2, e2):
    return max(s1, s2) < min(e1, e2)


In [21]:
doctor_availability = {}

for _, row in Doctors.iterrows():

    inst = row["Instructor_ID"]
    day = row["Day"]

    start = time_to_minutes(row["Start_Time"])
    end = time_to_minutes(row["End_Time"])

    doctor_availability.setdefault(inst, {})
    doctor_availability[inst].setdefault(day, [])
    doctor_availability[inst][day].append((start, end))

days = Doctors["Day"].unique().tolist()
time_slots = list(range(8 * 60, 17 * 60 + 1, 60))

print("Doctor availability ready")


Doctor availability ready


In [22]:
class SchedulingILP:

    def __init__(self, Courses, Rooms, Divisions):
        self.Courses = Courses
        self.Rooms = Rooms
        self.Divisions = Divisions
        self.prepare_pairs()

    def prepare_pairs(self):

        self.pairs = []

        for _, course in self.Courses.iterrows():

            matching = self.Divisions[
                (self.Divisions["Year"] == course["Year"]) &
                (self.Divisions["Major"] == course["Major"]) &
                (self.Divisions["Department"] == course["Department"])
            ]

            for _, div in matching.iterrows():
                self.pairs.append((course, div))

    def run(self):

        solver = pywraplp.Solver.CreateSolver("CBC")
        x = {}

        # VARIABLES
        for course, div in self.pairs:

            duration = course["Hours_per_day"] * 60
            instructor = course["Instructor_ID"]

            for day in days:
                for start in time_slots:

                    end = start + duration
                    if end > 17*60:
                        continue

                    for _, room in self.Rooms.iterrows():

                        if room["Type"] != course["Type"]:
                            continue

                        if room["Capacity"] < div["StudentNum"]:
                            continue

                        key = (
                            course["Course_ID"],
                            div["Num_ID"],
                            instructor,
                            day,
                            start,
                            end,
                            room["Room_ID"]
                        )

                        x[key] = solver.IntVar(0, 1, str(key))

        print("Variables:", len(x))

        # REQUIRED DAYS (≤ مش == عشان يطلع جدول)
        for course, div in self.pairs:

            required = course["Days"]

            relevant = [
                var for key, var in x.items()
                if key[0] == course["Course_ID"]
                and key[1] == div["Num_ID"]
            ]

            if relevant:
                solver.Add(solver.Sum(relevant) <= required)

        # CONFLICTS
        keys = list(x.keys())

        for i in range(len(keys)):
            for j in range(i+1, len(keys)):

                k1 = keys[i]
                k2 = keys[j]

                if k1[3] != k2[3]:
                    continue

                if not overlap(k1[4], k1[5], k2[4], k2[5]):
                    continue

                # Room conflict
                if k1[6] == k2[6]:
                    solver.Add(x[k1] + x[k2] <= 1)

                # Instructor conflict
                if k1[2] == k2[2]:
                    solver.Add(x[k1] + x[k2] <= 1)

                # Division conflict
                if k1[1] == k2[1]:
                    solver.Add(x[k1] + x[k2] <= 1)

        solver.Maximize(solver.Sum(x.values()))
        solver.SetTimeLimit(60000)

        status = solver.Solve()

        if status not in (pywraplp.Solver.OPTIMAL, pywraplp.Solver.FEASIBLE):
            print("No Solution Found")
            return []

        schedule = []

        for key, var in x.items():
            if var.solution_value() == 1:
                schedule.append(key)

        return schedule


In [23]:
ilp = SchedulingILP(Courses, Rooms, Divisions)
result = ilp.run()

print("Assignments:", len(result))


Variables: 4104
Assignments: 84


In [24]:
schedule_data = []

for r in result:

    course_id, div_id, instructor, day, start, end, room_id = r

    course_info = Courses[Courses["Course_ID"] == course_id].iloc[0]
    instructor_info = Doctors[Doctors["Instructor_ID"] == instructor].iloc[0]
    division_info = Divisions[Divisions["Num_ID"] == div_id].iloc[0]
    room_info = Rooms[Rooms["Room_ID"] == room_id].iloc[0]

    schedule_data.append({
        "Day": day,
        "Course_Name": course_info["Course_Name"],
        "Instructor_Name": instructor_info["Instructor_Name"],
        "Students": division_info["StudentNum"],
        "Room": room_info["Room"],
        "Start_Time": start,
        "End_Time": end,
        "Department": course_info["Department"],
        "Major": course_info["Major"]
    })

Schedule = pd.DataFrame(schedule_data)

print(Schedule.head())
print("Total:", len(Schedule))


        Day      Course_Name Instructor_Name  Students       Room  Start_Time  \
0   Tuesday  Mathematics (0)       Dr. Amany       219  Terrace 8         600   
1    Sunday  Mathematics (0)       Dr. Amany       219  Terrace 8         540   
2  Saturday  Mathematics (1)       Dr. Amany       219  Terrace 8         540   
3    Sunday  Mathematics (1)       Dr. Amany       219  Terrace 8         660   
4  Saturday     Electronics      Dr. Ibrahim       219  Terrace 8         600   

   End_Time Department Major  
0       660         IT    IT  
1       600         IT    IT  
2       600         IT    IT  
3       720         IT    IT  
4       660         IT    IT  
Total: 84


In [25]:
OUTPUT_PATH = r"../data/Processed Data/Schedule_Output_ILP.xlsx"
Schedule.to_excel(OUTPUT_PATH, index=False)

print("Saved to:", OUTPUT_PATH)


Saved to: ../data/Processed Data/Schedule_Output_ILP.xlsx
